# 01 – Liander electricity consumption (small connections)

**Goal:** turn the raw Liander consumption files (2019–2026) into one clean table.
**Output:** `Data/processed/liander_electricity_2019_2026.csv`

## Findings
- Every raw file has a different format (tab vs `;`, decimal comma vs dot, lines wrapped in quotes, extra spaces) → one reader function `read_liander()` handles all of them.
- The "Collected" Excel file is **not used**: decimal commas were lost (36 connections became 3600).
- Only electricity (`ELK`) is kept.
- Data is the Standard Yearly Consumption (SJV): *expected* use based on the previous year, not measured use. Reference date: 1 January of the file year.
- Small connections (≤ 3x80 A) include small businesses, so the average (~4,000 kWh) is higher than for homes only (~2,700 kWh, CBS).
- 3x25 connections (42%) use ~5,200 kWh, compared with ~2,500–2,900 for 1x25/1x35 → possible electrification feature.

## Break in the series (2024 → 2025)
- From 2025 Liander only counts connections that are **in operation**. Active connections in 2024 (3.33 M) vs 2025 (3.36 M) = +1.1% → connection drop explained.
- Total consumption drops 18%, only partly explained by this → the rest is unexplained.
- **Decision:** treat 2019–2024 and 2025–2026 as two series. Model on year-to-year **growth within a series**; exclude the 2024→2025 pair. Forecast 2026 → 2027.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../Data/raw")
PROCESSED = Path("../Data/processed")
CONS = RAW / "4 Liander" / "Raw Data" / "2. Consumption data for small-scale connections"

for f in sorted(CONS.glob("*.csv")):
    print(f.name)

Liander_kleinverbruiksgegevens_01012019.csv
Liander_kleinverbruiksgegevens_01012020.csv
Liander_kleinverbruiksgegevens_20210101.csv
Liander_kleinverbruiksgegevens_20220101.csv
Liander_kleinverbruiksgegevens_20230101.csv
Liander_kleinverbruiksgegevens_20240101.csv
Verbruiksdata KV 2025in.csv
Verbruiksdata KV 2026.csv


In [2]:
for f in sorted(CONS.glob("*.csv")):
    with open(f, encoding="utf-8-sig", errors="replace") as file:
        header = file.readline().strip()
        first_row = file.readline().strip()
    print("=" * 60)
    print(f.name)
    print("HEADER:", header[:150])
    print("ROW   :", first_row[:150])
    

Liander_kleinverbruiksgegevens_01012019.csv
HEADER: "NETBEHEERDER"	"NETGEBIED"	"STRAATNAAM"	"POSTCODE_VAN"	"POSTCODE_TOT"	"WOONPLAATS"	"LANDCODE"	"PRODUCTSOORT"	"VERBRUIKSSEGMENT"	"AANSLUITINGEN_AANTAL"
ROW   : "Liander NB"	"LIANDER"	"De Ruyterkade Steigers"	"1011AA"	"1011AB"	"AMSTERDAM"	"NL"	"ELK"	"KVB"	36,00	100,00	38,89	100,00	"3x25"	16589,00	44,44	25,00
Liander_kleinverbruiksgegevens_01012020.csv
HEADER: NETBEHEERDER	NETGEBIED	STRAATNAAM	POSTCODE_VAN	POSTCODE_TOT	WOONPLAATS	LANDCODE	PRODUCTSOORT	VERBRUIKSSEGMENT	AANSLUITINGEN_AANTAL	LEVERINGSRICHTING_P
ROW   : "Liander NB"	"LIANDER"	"De Ruyterkade Steigers"	"1011AA"	"1011AB"	"AMSTERDAM"	"NL"	"ELK"	"KVB"	43,00	100,00	32,56	37,21	"3x25"	14940	53,49	20,93
Liander_kleinverbruiksgegevens_20210101.csv
HEADER: NETBEHEERDER	NETGEBIED	STRAATNAAM	POSTCODE_VAN	POSTCODE_TOT	WOONPLAATS	LANDCODE	PRODUCTSOORT	VERBRUIKSSEGMENT	AANSLUITINGEN_AANTAL	LEVERINGSRICHTING_P
ROW   : "Liander NB"	"LIANDER"	"De Ruyterkade Steigers"	"1011AA"	"1011AB"	"AMSTER

In [3]:
from io import StringIO

def read_liander(path):
    # 1. read the file as plain text lines
    with open(path, encoding="utf-8-sig", errors="replace") as file:
        lines = [line.strip() for line in file.read().splitlines() if line.strip()]

    header = lines[0]

    # 2. detect the separator: tab or semicolon
    sep = "\t" if "\t" in header else ";"

    # 3. detect "whole line wrapped in quotes" (2022, 2025, 2026)
    wrapped = header.startswith('"') and header.endswith('"') and header.count('"') == 2
    if wrapped:
        lines = [line[1:-1].replace('""', '"') if line.startswith('"') and line.endswith('"') else line
                 for line in lines]

    # 4. let pandas read the cleaned text; keep everything as text for now
    df = pd.read_csv(StringIO("\n".join(lines)), sep=sep, dtype=str)

    # 5. remove extra spaces in column names and values
    df.columns = df.columns.str.strip()
    for col in df.columns:
        df[col] = df[col].str.strip()
    return df


# test it on every file
for f in sorted(CONS.glob("*.csv")):
    df = read_liander(f)
    print("=" * 60)
    print(f.name, "->", df.shape)
    print("columns:", df.columns.tolist())
    print("first row:", df.iloc[0].tolist())

Liander_kleinverbruiksgegevens_01012019.csv -> (261854, 17)
columns: ['NETBEHEERDER', 'NETGEBIED', 'STRAATNAAM', 'POSTCODE_VAN', 'POSTCODE_TOT', 'WOONPLAATS', 'LANDCODE', 'PRODUCTSOORT', 'VERBRUIKSSEGMENT', 'AANSLUITINGEN_AANTAL', 'LEVERINGSRICHTING_PERC', 'FYSIEKE_STATUS_PERC', 'SOORT_AANSLUITING_PERC', 'SOORT_AANSLUITING', 'SJV_GEMIDDELD', 'SJV_LAAG_TARIEF_PERC', 'SLIMME_METER_PERC']
first row: ['Liander NB', 'LIANDER', 'De Ruyterkade Steigers', '1011AA', '1011AB', 'AMSTERDAM', 'NL', 'ELK', 'KVB', '36,00', '100,00', '38,89', '100,00', '3x25', '16589,00', '44,44', '25,00']
Liander_kleinverbruiksgegevens_01012020.csv -> (262919, 17)
columns: ['NETBEHEERDER', 'NETGEBIED', 'STRAATNAAM', 'POSTCODE_VAN', 'POSTCODE_TOT', 'WOONPLAATS', 'LANDCODE', 'PRODUCTSOORT', 'VERBRUIKSSEGMENT', 'AANSLUITINGEN_AANTAL', 'LEVERINGSRICHTING_PERC', 'FYSIEKE_STATUS_PERC', 'SOORT_AANSLUITING_PERC', 'SOORT_AANSLUITING', 'SJV_GEMIDDELD', 'SJV_LAAG_TARIEF_PERC', 'SLIMME_METER_PERC']
first row: ['Liander NB', 'LIA

In [4]:
import re

NUM_COLS = ["AANSLUITINGEN_AANTAL", "LEVERINGSRICHTING_PERC", "SOORT_AANSLUITING_PERC",
            "SJV_GEMIDDELD", "SJV_LAAG_TARIEF_PERC", "SLIMME_METER_PERC"]
KEEP_COLS = ["POSTCODE_VAN", "POSTCODE_TOT", "WOONPLAATS", "SOORT_AANSLUITING"] + NUM_COLS

old_files = [f for f in sorted(CONS.glob("Liander_kleinverbruiksgegevens_*.csv"))]
parts = []

for f in old_files:
    df = read_liander(f)
    df = df[df["PRODUCTSOORT"] == "ELK"]          # electricity only
    df = df[KEEP_COLS].copy()                     # keep only the columns we need
    for col in NUM_COLS:                          # "36,00" -> 36.0
        df[col] = pd.to_numeric(df[col].str.replace(",", ".", regex=False), errors="coerce")
    df["year"] = int(re.search(r"(20\d{2})", f.name).group(1))   # year from file name
    parts.append(df)
    print(f.name, "->", len(df), "electricity rows")

elec_old = pd.concat(parts, ignore_index=True)
print("\nCombined:", elec_old.shape)

Liander_kleinverbruiksgegevens_01012019.csv -> 143131 electricity rows
Liander_kleinverbruiksgegevens_01012020.csv -> 143886 electricity rows
Liander_kleinverbruiksgegevens_20210101.csv -> 144620 electricity rows
Liander_kleinverbruiksgegevens_20220101.csv -> 145410 electricity rows
Liander_kleinverbruiksgegevens_20230101.csv -> 146191 electricity rows
Liander_kleinverbruiksgegevens_20240101.csv -> 149524 electricity rows

Combined: (872762, 11)


In [5]:
# 1. missing values per column
print("Missing values per column:")
print(elec_old.isna().sum())

# 2. summary per year
summary = elec_old.groupby("year").agg(
    rows=("POSTCODE_VAN", "size"),
    total_connections=("AANSLUITINGEN_AANTAL", "sum"),
    avg_kwh_per_connection=("SJV_GEMIDDELD", "mean"),
    avg_smart_meter_pct=("SLIMME_METER_PERC", "mean"),
    avg_consume_only_pct=("LEVERINGSRICHTING_PERC", "mean"),
).round(1)
print("\nSummary per year:")
print(summary)

Missing values per column:
POSTCODE_VAN              0
POSTCODE_TOT              0
WOONPLAATS                0
SOORT_AANSLUITING         0
AANSLUITINGEN_AANTAL      0
LEVERINGSRICHTING_PERC    0
SOORT_AANSLUITING_PERC    0
SJV_GEMIDDELD             0
SJV_LAAG_TARIEF_PERC      0
SLIMME_METER_PERC         0
year                      0
dtype: int64

Summary per year:
        rows  total_connections  avg_kwh_per_connection  avg_smart_meter_pct  \
year                                                                           
2019  143131          3460267.0                  4172.2                 55.2   
2020  143886          3505527.0                  4165.8                 69.7   
2021  144620          3550671.0                  4154.9                 78.0   
2022  145410          3600776.0                  4164.5                 80.8   
2023  146191          3649849.0                  4013.8                 81.0   
2024  149524          3699495.0                  3920.5                 8

In [6]:
# total yearly consumption of each row = connections x average per connection
elec_old["total_kwh"] = elec_old["AANSLUITINGEN_AANTAL"] * elec_old["SJV_GEMIDDELD"]

per_year = elec_old.groupby("year").agg(
    total_connections=("AANSLUITINGEN_AANTAL", "sum"),
    total_kwh=("total_kwh", "sum"),
)
per_year["weighted_kwh_per_connection"] = per_year["total_kwh"] / per_year["total_connections"]
per_year["total_TWh"] = per_year["total_kwh"] / 1e9

print(per_year[["total_connections", "total_TWh", "weighted_kwh_per_connection"]].round(2))

      total_connections  total_TWh  weighted_kwh_per_connection
year                                                           
2019          3460267.0      14.39                      4159.92
2020          3505527.0      14.56                      4152.48
2021          3550671.0      14.72                      4146.88
2022          3600776.0      14.98                      4159.22
2023          3649849.0      14.67                      4018.84
2024          3699495.0      14.33                      3874.26


In [7]:
check = elec_old[elec_old["year"] == 2024].groupby("SOORT_AANSLUITING").agg(
    rows=("POSTCODE_VAN", "size"),
    connections=("AANSLUITINGEN_AANTAL", "sum"),
    total_kwh=("total_kwh", "sum"),
)
check["kwh_per_connection"] = (check["total_kwh"] / check["connections"]).round(0)
check["share_of_connections_pct"] = (100 * check["connections"] / check["connections"].sum()).round(1)

print(check.sort_values("connections", ascending=False)[["rows", "share_of_connections_pct", "kwh_per_connection"]].head(12))

                    rows  share_of_connections_pct  kwh_per_connection
SOORT_AANSLUITING                                                     
3x25               61289                      42.3              5193.0
1x35               60637                      40.6              2873.0
1x25               26436                      16.4              2465.0
3x35                 469                       0.3             14721.0
3x80                 380                       0.2             23522.0
3x63                  94                       0.0             19436.0
1x6                   70                       0.0              9391.0
3x50                  63                       0.0             17115.0
OBK                   39                       0.0              7805.0
1x50                  21                       0.0              2379.0
1x10                  25                       0.0              8231.0
1x63                   1                       0.0              3084.0


In [8]:
new_files = sorted(CONS.glob("Verbruiksdata KV *.csv"))
new_parts = []

for f in new_files:
    df = read_liander(f)
    df = df[df["PRODUCTSOORT"] == "ELK"].copy()
    for col in ["AANSLUITINGEN_AANTAL", "POSTCODE_AANTAL", "TOT_E", "SJA GEMIDDELD"]:
        df[col] = pd.to_numeric(df[col].str.replace(",", ".", regex=False), errors="coerce")
    df = df.rename(columns={
        "POSTCODE": "POSTCODE_VAN",
        "POSTCODE_EIND": "POSTCODE_TOT",
        "TYPE_AANSLUITING_MEEST_VOORKOMEND": "SOORT_AANSLUITING",
        "SJA GEMIDDELD": "SJV_GEMIDDELD",
        "TOT_E": "total_kwh",
    })
    df["year"] = int(re.search(r"(20\d{2})", f.name).group(1))
    new_parts.append(df)
    print(f.name, "->", len(df), "electricity rows")

elec_new = pd.concat(new_parts, ignore_index=True)

# check 1: is total_kwh really connections x average?
ratio = elec_new["total_kwh"] / (elec_new["AANSLUITINGEN_AANTAL"] * elec_new["SJV_GEMIDDELD"])
print("\nMedian of total / (connections x avg):", round(ratio.median(), 3))

# check 2: compare totals with the old years
per_year_new = elec_new.groupby("year").agg(
    total_connections=("AANSLUITINGEN_AANTAL", "sum"),
    total_kwh=("total_kwh", "sum"),
)
per_year_new["weighted_kwh_per_connection"] = per_year_new["total_kwh"] / per_year_new["total_connections"]
per_year_new["total_TWh"] = per_year_new["total_kwh"] / 1e9

both = pd.concat([per_year, per_year_new])
print(both[["total_connections", "total_TWh", "weighted_kwh_per_connection"]].round(2))

Verbruiksdata KV 2025in.csv -> 147280 electricity rows
Verbruiksdata KV 2026.csv -> 147770 electricity rows

Median of total / (connections x avg): 1.0
      total_connections  total_TWh  weighted_kwh_per_connection
year                                                           
2019          3460267.0      14.39                      4159.92
2020          3505527.0      14.56                      4152.48
2021          3550671.0      14.72                      4146.88
2022          3600776.0      14.98                      4159.22
2023          3649849.0      14.67                      4018.84
2024          3699495.0      14.33                      3874.26
2025          3361876.0      11.70                      3480.21
2026          3397203.0      12.07                      3552.84


In [9]:
f2024 = CONS / "Liander_kleinverbruiksgegevens_20240101.csv"
t = read_liander(f2024)
t = t[t["PRODUCTSOORT"] == "ELK"].copy()
for col in ["AANSLUITINGEN_AANTAL", "FYSIEKE_STATUS_PERC", "SJV_GEMIDDELD"]:
    t[col] = pd.to_numeric(t[col].str.replace(",", ".", regex=False), errors="coerce")

all_conn = t["AANSLUITINGEN_AANTAL"].sum()
active_conn = (t["AANSLUITINGEN_AANTAL"] * t["FYSIEKE_STATUS_PERC"] / 100).sum()

print(f"2024 old file, all connections:    {all_conn:,.0f}")
print(f"2024 old file, active connections: {active_conn:,.0f}")
print(f"2025 new file, connections:        {per_year_new.loc[2025, 'total_connections']:,.0f}")
print(f"Average % active in 2024:          {t['FYSIEKE_STATUS_PERC'].mean():.1f}%")

2024 old file, all connections:    3,699,495
2024 old file, active connections: 3,326,465
2025 new file, connections:        3,361,876
Average % active in 2024:          91.1%


In [10]:
elec_old["series"] = "old_2019_2024"
elec_new["series"] = "new_2025_2026"

cols = ["year", "series", "POSTCODE_VAN", "POSTCODE_TOT", "SOORT_AANSLUITING",
        "AANSLUITINGEN_AANTAL", "SJV_GEMIDDELD", "total_kwh",
        "LEVERINGSRICHTING_PERC", "SLIMME_METER_PERC", "SJV_LAAG_TARIEF_PERC", "POSTCODE_AANTAL"]

elec_all = pd.concat([elec_old, elec_new], ignore_index=True)[cols]

PROCESSED.mkdir(parents=True, exist_ok=True)
out_file = PROCESSED / "liander_electricity_2019_2026.csv"
elec_all.to_csv(out_file, index=False)

print("Saved:", out_file, elec_all.shape)
print(elec_all.groupby(["series", "year"]).size())

Saved: ..\Data\processed\liander_electricity_2019_2026.csv (1167812, 12)
series         year
new_2025_2026  2025    147280
               2026    147770
old_2019_2024  2019    143131
               2020    143886
               2021    144620
               2022    145410
               2023    146191
               2024    149524
dtype: int64
